In [1]:
import pandas as pd
import pickle
import os
from nltk.tokenize import sent_tokenize
import nltk
import re
import string
# import gensim
import itertools
import matplotlib.pyplot as plt 
nltk.download('punkt')
import bs4 as bs
import numpy as np
import json
import ast
from functools import reduce
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

directory = '../data/'
translator = re.compile('[%s]' % re.escape(string.punctuation))
plt.rcParams['figure.figsize'] = (10,7)
plt.style.use('ggplot')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\xtang2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
from openai import OpenAI
client = OpenAI(api_key="sk-proj-fyoBu7GzQG8H6LEq0lPdT3BlbkFJfZqc0U4ZYoIy89R6I2VP")
# client = OpenAI(api_key='sk-None-eGR9kyQrLnY0BzZtfI67T3BlbkFJt8kbEsatG9t8dFMIF4hJ')
# client = OpenAI(api_key="sk-proj-5qzZVEalSawZKz3tQLU1T3BlbkFJW3JlMmi6EKwlqyiNyFT4") # mine

In [6]:
type_dict = {'staff': 'IMF staff', 'buff': 'country\'s authority'}

1. processing training and testing sets

In [11]:
# prepare files for different tasks
df_train = pd.read_excel(directory+'output/finetuning/training2.xlsx')
df_test = pd.read_excel(directory+'output/finetuning/testing2.xlsx')

df_train_stance = pd.DataFrame()
for tp in ['staff', 'buff']:
    df_temp = df_train[['index', 'Print ISBN', 'country', 'year', tp, '%s_stance_current'%tp, '%s_stance_future'%tp]]
    df_temp = df_temp.rename(columns={tp: 'text'}).rename(columns={c: c.replace(tp+'_', '') for c in df_temp.columns})
    df_temp['type'] = tp
    df_train_stance = pd.concat([df_train_stance, df_temp], ignore_index=True)
    
df_test_stance = pd.DataFrame()
for tp in ['staff', 'buff']:
    df_temp = df_test[['index', 'Print ISBN', 'country', 'year', tp, '%s_stance_current'%tp, '%s_stance_future'%tp]]
    df_temp = df_temp.rename(columns={tp: 'text'}).rename(columns={c: c.replace(tp+'_', '') for c in df_temp.columns})
    df_temp['type'] = tp
    df_test_stance = pd.concat([df_test_stance, df_temp], ignore_index=True)
    
# df_train_agree = df_train[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year', 'agreement_general', 'disagreement_areas']]
# df_test_agree = df_test[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year', 'agreement_general', 'disagreement_areas']]

In [13]:
for col in ['stance_current', 'stance_future']:
    print(df_train_stance[col].value_counts(normalize=True))
    print(df_test_stance[col].value_counts(normalize=True))
    print('\n')

stance_current
accommodative    0.376623
unclear          0.344156
restrictive      0.229437
irrelevant       0.032468
neutral          0.017316
Name: proportion, dtype: float64
stance_current
unclear          0.448276
accommodative    0.344828
restrictive      0.129310
irrelevant       0.060345
neutral          0.017241
Name: proportion, dtype: float64


stance_future
no change          0.482684
tightening bias    0.177489
tightening         0.134199
loosening bias     0.080087
unclear            0.051948
loosening          0.041126
irrelevant         0.032468
Name: proportion, dtype: float64
stance_future
no change          0.405172
loosening bias     0.146552
tightening bias    0.129310
tightening         0.120690
unclear            0.086207
loosening          0.060345
irrelevant         0.051724
Name: proportion, dtype: float64




2. Zero-shot tests - short/long queries/chain-of-thought

In [7]:
type_dict = {'staff': 'IMF staff', 'buff': 'country\'s authority'}

In [75]:
# version 1: simplest
for i, row in df_test_stance.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year written by %s, complete the following two tasks. First, classify the country\'s recent or current monetary policy stance as described in the text into restrictive/neutral/accommodative/unclear/irrelevant; if it discusses monetary policy but the specific stance is not clear, assign unclear; if it does not discuss monetary policy, assign irrelevant. Second, classify the %s\'s recommended or planned near-future (next year) direction of change in monetary policy stance as described in the text into tightening/tightening bias/no change/loosening bias/loosening/unclear/irrelevant; if it discusses monetary policy stance but the direction of change is not clear, assign no change; if it does not discuss monetary policy stance, assign unclear (if it discusses monetary policy) or irrelevant (if it does not discuss monetary policy). Return a JSON dict without additional texts: \"stance_current\", \"stance_future\".'''%(type_dict[row['type']],  type_dict[row['type']]),},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\nText:\n%s''' % (row['country'], row['year'], row['text'])}
        ],
            model='gpt-4o-2024-08-06', 
            temperature=0
        )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```',''))
        df_test_stance.loc[i,'stance_current_gpt1'] = result['stance_current']
        df_test_stance.loc[i,'stance_future_gpt1'] = result['stance_future']
    except Exception:
        df_test_stance.loc[i, 'error_gpt1'] = chat_completion.choices[0].message.content

In [96]:
# query v1:
'raw:', accuracy_score(df_test_stance['stance_current'], df_test_stance['stance_current_gpt1']), accuracy_score(df_test_stance['stance_future'], df_test_stance['stance_future_gpt1']), f1_score(df_test_stance['stance_current'], df_test_stance['stance_current_gpt1'], average='weighted'), f1_score(df_test_stance['stance_future'], df_test_stance['stance_future_gpt1'], average='weighted'), 'merging unclear/irrelevant:', \
accuracy_score(df_test_stance['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_current_gpt1'].apply(lambda x: 'unclear' if x=='irrelevant' else x)), accuracy_score(df_test_stance['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_future_gpt1'].apply(lambda x: 'unclear' if x=='irrelevant' else x)), f1_score(df_test_stance['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_current_gpt1'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted'), f1_score(df_test_stance['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_future_gpt1'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted')

('raw:',
 0.49137931034482757,
 0.6637931034482759,
 0.41065360631106707,
 0.6340632211063693,
 'merging unclear/irrelevant:',
 0.6293103448275862,
 0.7327586206896551,
 0.6540910640742431,
 0.7218375157772157)

In [103]:
# version 2: adding brief definitions
for i, row in df_test_stance.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year written by %s, complete the following two tasks.\n\nFirst, classify the country\'s recent or current monetary policy stance as described in the text into restrictive/neutral/accommodative/unclear/irrelevant. Definitions:\nRestrictive: The policy aims to reduce inflation and prevent the economy from overheating. This is typically achieved through higher interest rates or other measures that reduce the money supply.\nNeutral: The policy neither aims to reduce inflation nor stimulate the economy. It is intended to maintain the current economic conditions without significant changes.\nAccommodative: The policy aims to stimulate the economy, usually to combat unemployment or to encourage growth. This often involves lower interest rates or measures to increase the money supply.\nUnclear: The text discusses monetary policy but the specific stance is not clear.\nIrrelevant: The text does not discuss monetary policy.\n\nSecond, classify the %s\'s recommended or planned near-future (next year) direction of change in monetary policy stance as described in the text into tightening/tightening bias/no change/loosening bias/loosening/unclear/irrelevant. Definitions:\nTightening: Suggests a plan to make the policy more restrictive, typically to combat inflation or overheating in the economy.\nTightening Bias: Indicates a leaning towards tightening, though not explicitly planning to do so.\nNo Change: Indicates a plan to maintain the current policy stance, or an unclear policy direction if the text discusses monetary policy stance.\nLoosening Bias: Suggests a leaning towards loosening, though not explicitly planning to do so.\nLoosening: Suggests a plan to make the policy more accommodative, typically to stimulate economic growth or combat unemployment.\nUnclear: The text discusses monetary policy but does not discuss monetary policy stance.\nIrrelevant: The text does not discuss monetary policy.\n\nReturn a JSON dict without additional texts: \"stance_current\", \"stance_future\".'''%(type_dict[row['type']],  type_dict[row['type']]),},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\nText:\n%s''' % (row['country'], row['year'], row['text'])}
        ],
            model='gpt-4o-2024-08-06', 
            temperature=0
        )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```',''))
        df_test_stance.loc[i,'stance_current_gpt2'] = result['stance_current']
        df_test_stance.loc[i,'stance_future_gpt2'] = result['stance_future']
    except Exception:
        df_test_stance.loc[i, 'error_gpt2'] = chat_completion.choices[0].message.content

In [106]:
# query v2:
'raw:', accuracy_score(df_test_stance['stance_current'], df_test_stance['stance_current_gpt2']), accuracy_score(df_test_stance['stance_future'], df_test_stance['stance_future_gpt2']), f1_score(df_test_stance['stance_current'], df_test_stance['stance_current_gpt2'], average='weighted'), f1_score(df_test_stance['stance_future'], df_test_stance['stance_future_gpt2'], average='weighted'), 'merging unclear/irrelevant:', \
accuracy_score(df_test_stance['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_current_gpt2'].apply(lambda x: 'unclear' if x=='irrelevant' else x)), accuracy_score(df_test_stance['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_future_gpt2'].apply(lambda x: 'unclear' if x=='irrelevant' else x)), f1_score(df_test_stance['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_current_gpt2'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted'), f1_score(df_test_stance['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_future_gpt2'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted')

('raw:',
 0.47413793103448276,
 0.6120689655172413,
 0.4086044480441031,
 0.5902333236233933,
 'merging unclear/irrelevant:',
 0.5344827586206896,
 0.6637931034482759,
 0.529224544741786,
 0.6511528638532783)

In [111]:
df_test_stance[['stance_current','stance_current_gpt2']].value_counts()

stance_current  stance_current_gpt2
accommodative   accommodative          37
unclear         neutral                25
restrictive     restrictive            12
unclear         accommodative          11
                restrictive             9
                irrelevant              6
irrelevant      irrelevant              3
restrictive     neutral                 3
accommodative   neutral                 2
irrelevant      accommodative           2
neutral         neutral                 2
accommodative   restrictive             1
irrelevant      neutral                 1
                unclear                 1
unclear         unclear                 1
dtype: int64

In [112]:
# version 3: chain-of-thought
for i, row in df_test_stance.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year written by %s, complete the following two tasks. First, classify the country\'s recent or current monetary policy stance as described in the text into restrictive/neutral/accommodative/unclear/irrelevant; if it discusses monetary policy but the specific stance is not clear, assign unclear; if it does not discuss monetary policy, assign irrelevant. Second, classify the %s\'s recommended or planned near-future (next year) direction of change in monetary policy stance as described in the text into tightening/tightening bias/no change/loosening bias/loosening/unclear/irrelevant; if it discusses monetary policy stance but the direction of change is not clear, assign no change; if it does not discuss monetary policy stance, assign unclear (if it discusses monetary policy) or irrelevant (if it does not discuss monetary policy). Provide reasoning for your classifications. Return a JSON dict without additional texts: \"stance_current\", \"stance_future\", \"reason\".'''%(type_dict[row['type']],  type_dict[row['type']]),},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\nText:\n%s''' % (row['country'], row['year'], row['text'])}
        ],
            model='gpt-4o-2024-08-06', 
            temperature=0
        )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```',''))
        df_test_stance.loc[i,'stance_current_gpt3'] = result['stance_current']
        df_test_stance.loc[i,'stance_future_gpt3'] = result['stance_future']
        df_test_stance.loc[i,'reason_gpt3'] = result['reason']
    except Exception:
        df_test_stance.loc[i, 'error_gpt3'] = chat_completion.choices[0].message.content

In [113]:
# query v3:
'raw:', accuracy_score(df_test_stance['stance_current'], df_test_stance['stance_current_gpt3']), accuracy_score(df_test_stance['stance_future'], df_test_stance['stance_future_gpt3']), f1_score(df_test_stance['stance_current'], df_test_stance['stance_current_gpt3'], average='weighted'), f1_score(df_test_stance['stance_future'], df_test_stance['stance_future_gpt3'], average='weighted'), 'merging unclear/irrelevant:', \
accuracy_score(df_test_stance['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_current_gpt3'].apply(lambda x: 'unclear' if x=='irrelevant' else x)), accuracy_score(df_test_stance['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_future_gpt3'].apply(lambda x: 'unclear' if x=='irrelevant' else x)), f1_score(df_test_stance['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_current_gpt3'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted'), f1_score(df_test_stance['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_future_gpt3'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted')

('raw:',
 0.4827586206896552,
 0.6379310344827587,
 0.4078289370346712,
 0.6113387914066523,
 'merging unclear/irrelevant:',
 0.6293103448275862,
 0.7155172413793104,
 0.6496339604488901,
 0.7057419293962313)

3. Few-shot

In [181]:
# df_train_stance['len_all'] = df_train_stance.apply(lambda x: (x['text']).count(' '), axis=1)
# df_train_stance[(df_train_stance['type']=='buff')&(df_train_stance['stance_future']=='loosening')].sort_values(by=['len_all']).iloc[2]#['text']

In [182]:
# version 4: adding a few examples
example_dict = {'staff': 'Example 1: Country: Guyana; Year: 2017; Text: \"The accommodative monetary policy is appropriate, but should gradually move towards a neutral stance in 2017. Pass-through from the exchange rate and from the VAT reform to inflation should be closely monitored.\" Answer: {\"stance_current\": \"accommodative\", \"stance_future\": \"tightening\"}.\nExample 2: Country: Mauritius; Year: 2015; Text: \"The monetary policy stance is broadly appropriate given the low-inflation environment. Further excess liquidity absorption should proceed at a measured pace in order to avoid any sharp increases in market interest rates.\" Answer: {\"stance_current\": \"unclear\", \"stance_future\": \"no change\"}.\nExample 3: Country: Trinidad and Tobago; Year: 2017; Text: \"The current monetary policy is appropriate, and in any case, room for maneuver is limited. Modest interest rate easing could eventually support a recovery, but would be contingent on reestablishing policy credibility with a strong fiscal package, wide-ranging structural reforms, and restoring balance in the f/x market.\" Answer: {\"stance_current\": \"unclear\", \"stance_future\": \"loosening bias\"}.',
               'buff': 'Example 1: Country: Guyana; Year: 2019; Text: \"Monetary policy remained broadly accommodative in 2018. The Bank of Guyana (BoG) maintained a bank rate of 5 percent, whilst ensuring an adequate level of liquidity in the banking system to create an enabling environment for credit and economic growth.\" Answer: {\"stance_current\": \"accommodative\", \"stance_future\": \"no change\"}.\nExample 2: Country: Bangladesh; Year: 2018; Text: \"The monetary policy stance will remain prudent, and the authorities are vigilant against upside risks to inflation and ready for appropriate adjustments in both policy rates and reserve requirements.\" Answer: {\"stance_current\": \"unclear\", \"stance_future\": \"tightening bias\"}.\nExample 3: Country: Iran; Year: 2015; Text: \"Monetary policy is guided by the disinflation strategy which seeks to achieve single-digit inflation by end 2016/17. While prioritizing price stability over output growth, my authorities are of the view that some temporary relief to the economy is needed at present, given sluggish growth, better-than-expected inflation outturns, benign inflation outlook, and tight fiscal stance.\" Answer: {\"stance_current\": \"restrictive\", \"stance_future\": \"loosening\"}.'}
for i, row in df_test_stance.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year written by %s, complete the following two tasks. First, classify the country\'s recent or current monetary policy stance as described in the text into restrictive/neutral/accommodative/unclear/irrelevant; if it discusses monetary policy but the specific stance is not clear, assign unclear; if it does not discuss monetary policy, assign irrelevant. Second, classify the %s\'s recommended or planned near-future (next year) direction of change in monetary policy stance as described in the text into tightening/tightening bias/no change/loosening bias/loosening/unclear/irrelevant; if it discusses monetary policy stance but the direction of change is not clear, assign no change; if it does not discuss monetary policy stance, assign unclear (if it discusses monetary policy) or irrelevant (if it does not discuss monetary policy).\n%s\n\nReturn a JSON dict without additional texts: \"stance_current\", \"stance_future\".'''%(type_dict[row['type']],  type_dict[row['type']], example_dict[row['type']]),},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\nText:\n%s''' % (row['country'], row['year'], row['text'])}
        ],
            model='gpt-4o-2024-08-06', 
            temperature=0
        )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```',''))
        df_test_stance.loc[i,'stance_current_gpt4'] = result['stance_current']
        df_test_stance.loc[i,'stance_future_gpt4'] = result['stance_future']
    except Exception:
        df_test_stance.loc[i, 'error_gpt4'] = chat_completion.choices[0].message.content

In [183]:
# query v4:
'raw:', accuracy_score(df_test_stance['stance_current'], df_test_stance['stance_current_gpt4']), accuracy_score(df_test_stance['stance_future'], df_test_stance['stance_future_gpt4']), f1_score(df_test_stance['stance_current'], df_test_stance['stance_current_gpt4'], average='weighted'), f1_score(df_test_stance['stance_future'], df_test_stance['stance_future_gpt4'], average='weighted'), 'merging unclear/irrelevant:', \
accuracy_score(df_test_stance['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_current_gpt4'].apply(lambda x: 'unclear' if x=='irrelevant' else x)), accuracy_score(df_test_stance['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_future_gpt4'].apply(lambda x: 'unclear' if x=='irrelevant' else x)), f1_score(df_test_stance['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_current_gpt4'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted'), f1_score(df_test_stance['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_future_gpt4'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted')

('raw:',
 0.5172413793103449,
 0.6637931034482759,
 0.49472646457882646,
 0.6413458849964669,
 'merging unclear/irrelevant:',
 0.6724137931034483,
 0.7327586206896551,
 0.7070836080641898,
 0.7318167785819321)

4. other models

In [189]:
# version 5: gpt-4o-mini
for i, row in df_test_stance.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year written by %s, complete the following two tasks. First, classify the country\'s recent or current monetary policy stance as described in the text into restrictive/neutral/accommodative/unclear/irrelevant; if it discusses monetary policy but the specific stance is not clear, assign unclear; if it does not discuss monetary policy, assign irrelevant. Second, classify the %s\'s recommended or planned near-future (next year) direction of change in monetary policy stance as described in the text into tightening/tightening bias/no change/loosening bias/loosening/unclear/irrelevant; if it discusses monetary policy stance but the direction of change is not clear, assign no change; if it does not discuss monetary policy stance, assign unclear (if it discusses monetary policy) or irrelevant (if it does not discuss monetary policy). Return a JSON dict without additional texts: \"stance_current\", \"stance_future\".'''%(type_dict[row['type']],  type_dict[row['type']]),},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\nText:\n%s''' % (row['country'], row['year'], row['text'])}
        ],
            model='gpt-4o-mini-2024-07-18', 
            temperature=0
        )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```',''))
        df_test_stance.loc[i,'stance_current_gpt5'] = result['stance_current']
        df_test_stance.loc[i,'stance_future_gpt5'] = result['stance_future']
    except Exception:
        df_test_stance.loc[i, 'error_gpt5'] = chat_completion.choices[0].message.content

In [190]:
# query v5:
'raw:', accuracy_score(df_test_stance['stance_current'], df_test_stance['stance_current_gpt5']), accuracy_score(df_test_stance['stance_future'], df_test_stance['stance_future_gpt5']), f1_score(df_test_stance['stance_current'], df_test_stance['stance_current_gpt5'], average='weighted'), f1_score(df_test_stance['stance_future'], df_test_stance['stance_future_gpt5'], average='weighted'), 'merging unclear/irrelevant:', \
accuracy_score(df_test_stance['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_current_gpt5'].apply(lambda x: 'unclear' if x=='irrelevant' else x)), accuracy_score(df_test_stance['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_future_gpt5'].apply(lambda x: 'unclear' if x=='irrelevant' else x)), f1_score(df_test_stance['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_current_gpt5'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted'), f1_score(df_test_stance['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_future_gpt5'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted')

('raw:',
 0.47413793103448276,
 0.603448275862069,
 0.40153290701587985,
 0.5544723989490543,
 'merging unclear/irrelevant:',
 0.5344827586206896,
 0.6293103448275862,
 0.5016150088221195,
 0.593396118907257)

In [191]:
# version 6: gpt-3.5-turbo
for i, row in df_test_stance.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year written by %s, complete the following two tasks. First, classify the country\'s recent or current monetary policy stance as described in the text into restrictive/neutral/accommodative/unclear/irrelevant; if it discusses monetary policy but the specific stance is not clear, assign unclear; if it does not discuss monetary policy, assign irrelevant. Second, classify the %s\'s recommended or planned near-future (next year) direction of change in monetary policy stance as described in the text into tightening/tightening bias/no change/loosening bias/loosening/unclear/irrelevant; if it discusses monetary policy stance but the direction of change is not clear, assign no change; if it does not discuss monetary policy stance, assign unclear (if it discusses monetary policy) or irrelevant (if it does not discuss monetary policy). Return a JSON dict without additional texts: \"stance_current\", \"stance_future\".'''%(type_dict[row['type']],  type_dict[row['type']]),},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\nText:\n%s''' % (row['country'], row['year'], row['text'])}
        ],
            model='gpt-3.5-turbo', 
            temperature=0
        )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```',''))
        df_test_stance.loc[i,'stance_current_gpt6'] = result['stance_current']
        df_test_stance.loc[i,'stance_future_gpt6'] = result['stance_future']
    except Exception:
        df_test_stance.loc[i, 'error_gpt6'] = chat_completion.choices[0].message.content

In [192]:
# query v6:
'raw:', accuracy_score(df_test_stance['stance_current'], df_test_stance['stance_current_gpt6']), accuracy_score(df_test_stance['stance_future'], df_test_stance['stance_future_gpt6']), f1_score(df_test_stance['stance_current'], df_test_stance['stance_current_gpt6'], average='weighted'), f1_score(df_test_stance['stance_future'], df_test_stance['stance_future_gpt6'], average='weighted'), 'merging unclear/irrelevant:', \
accuracy_score(df_test_stance['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_current_gpt6'].apply(lambda x: 'unclear' if x=='irrelevant' else x)), accuracy_score(df_test_stance['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_future_gpt6'].apply(lambda x: 'unclear' if x=='irrelevant' else x)), f1_score(df_test_stance['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_current_gpt6'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted'), f1_score(df_test_stance['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_future_gpt6'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted')

('raw:',
 0.46551724137931033,
 0.5086206896551724,
 0.44841410319594566,
 0.3971965236893749,
 'merging unclear/irrelevant:',
 0.5,
 0.5086206896551724,
 0.5082732581255232,
 0.39931388606082074)

4. fine-tuning

In [12]:
# monetary classification
df_train_stance['answer'] = df_train_stance.apply(lambda x: "{'stance_current': '%s', 'stance_future': '%s'}" % (x['stance_current'], x['stance_future']), axis=1)
df_train_stance['messages'] = df_train_stance.apply(lambda row: [
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year expressing the views of %s, complete the following two tasks. First, classify the country\'s recent or current monetary policy stance as described in the text into restrictive/neutral/accommodative/unclear/irrelevant; if it discusses monetary policy but the specific stance is not clear, assign unclear; if it does not discuss monetary policy, assign irrelevant. Second, classify the %s\'s recommended or planned near-future (next year) direction of change in monetary policy stance as described in the text into tightening/tightening bias/no change/loosening bias/loosening/unclear/irrelevant; if it discusses monetary policy stance but the direction of change is not clear, assign no change; if it does not discuss monetary policy stance, assign unclear (if it discusses monetary policy) or irrelevant (if it does not discuss monetary policy). Return a JSON dict without additional texts: \"stance_current\", \"stance_future\".'''%(type_dict[row['type']],  type_dict[row['type']]),},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\nText:\n%s''' % (row['country'], row['year'], row['text'])},
                {   "role": "assistant",
                    "content":  row['answer']}
            ], axis=1)

df_train_stance[['messages']].to_json(directory+'output/finetuning/training_mon.jsonl', orient='records', lines=True)


In [8]:
# check data
import json
# import tiktoken # for token counting
import numpy as np
from collections import defaultdict

# Load the dataset
with open(directory+'output/finetuning/training_mon.jsonl', 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f]

# Initial dataset stats
print("Num examples:", len(dataset))
print("First example:")
for message in dataset[0]["messages"]:
    print(message)

Num examples: 462
First example:
{'role': 'system', 'content': 'You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year written by IMF staff, complete the following two tasks. First, classify the country\'s recent or current monetary policy stance as described in the text into restrictive/neutral/accommodative/unclear/irrelevant; if it discusses monetary policy but the specific stance is not clear, assign unclear; if it does not discuss monetary policy, assign irrelevant. Second, classify the IMF staff\'s recommended or planned near-future (next year) direction of change in monetary policy stance as described in the text into tightening/tightening bias/no change/loosening bias/loosening/unclear/irrelevant; if it discusses monetary policy stance but the direction of change is not clear, assign no change; if it does not discuss monetary policy stance, assign unclear (if it discusses monetary policy) or irrelevant (if it does no

In [9]:
# Format error checks
format_errors = defaultdict(int)

for ex in dataset:
    if not isinstance(ex, dict):
        format_errors["data_type"] += 1
        continue
        
    messages = ex.get("messages", None)
    if not messages:
        format_errors["missing_messages_list"] += 1
        continue
        
    for message in messages:
        if "role" not in message or "content" not in message:
            format_errors["message_missing_key"] += 1
        
        if any(k not in ("role", "content", "name", "function_call", "weight") for k in message):
            format_errors["message_unrecognized_key"] += 1
        
        if message.get("role", None) not in ("system", "user", "assistant", "function"):
            format_errors["unrecognized_role"] += 1
            
        content = message.get("content", None)
        function_call = message.get("function_call", None)
        
        if (not content and not function_call) or not isinstance(content, str):
            format_errors["missing_content"] += 1
    
    if not any(message.get("role", None) == "assistant" for message in messages):
        format_errors["example_missing_assistant_message"] += 1

if format_errors:
    print("Found errors:")
    for k, v in format_errors.items():
        print(f"{k}: {v}")
else:
    print("No errors found")

No errors found


In [198]:
# create finetuning job
file = client.files.create(
    file=open(directory+"output/finetuning/training_mon.jsonl", "rb"),
    purpose="fine-tune"
)
print(file.id)

file-OvhwTq86xHfWnpw3iZPeiPuy


In [251]:
client.fine_tuning.jobs.create(
    training_file=file.id,
    model="gpt-4o-mini-2024-07-18"
)

FineTuningJob(id='ftjob-1LJRrL3qutfiCvDuZWA2QCY2', created_at=1723494508, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs='auto', batch_size='auto', learning_rate_multiplier='auto'), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-3AqEuQZh0o1lNQSUihUxYYSd', result_files=[], seed=1712103162, status='validating_files', trained_tokens=None, training_file='file-OvhwTq86xHfWnpw3iZPeiPuy', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None)

In [258]:
# List up to 10 events from a fine-tuning job
client.fine_tuning.jobs.list_events(fine_tuning_job_id="ftjob-qOc0tsl2UDCXJaLMXrBKnjsC", limit=2)

SyncCursorPage[FineTuningJobEvent](data=[FineTuningJobEvent(id='ftevent-v1eTBQZTw3hf1GtaaxM0UBCW', created_at=1723469905, level='info', message='The job has successfully completed', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-SH9Ya0vkw0m0bhyxTqHCAYe3', created_at=1723469900, level='info', message='New fine-tuned model created: ft:gpt-4o-mini-2024-07-18:personal::9vPXXeG8', object='fine_tuning.job.event', data={}, type='message')], object='list', has_more=True)

In [210]:
model_id = 'ft:gpt-4o-mini-2024-07-18:personal::9vPXXeG8'

for i,row in df_test_stance.iterrows():
    chat_completion = client.chat.completions.create(
        messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year written by %s, complete the following two tasks. First, classify the country\'s recent or current monetary policy stance as described in the text into restrictive/neutral/accommodative/unclear/irrelevant; if it discusses monetary policy but the specific stance is not clear, assign unclear; if it does not discuss monetary policy, assign irrelevant. Second, classify the %s\'s recommended or planned near-future (next year) direction of change in monetary policy stance as described in the text into tightening/tightening bias/no change/loosening bias/loosening/unclear/irrelevant; if it discusses monetary policy stance but the direction of change is not clear, assign no change; if it does not discuss monetary policy stance, assign unclear (if it discusses monetary policy) or irrelevant (if it does not discuss monetary policy). Return a JSON dict without additional texts: \"stance_current\", \"stance_future\".'''%(type_dict[row['type']],  type_dict[row['type']]),},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\nText:\n%s''' % (row['country'], row['year'], row['text'])}
        ],
        model=model_id,
    )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```','').replace("\'", "\""))
        df_test_stance.loc[i,'stance_current_gpt_ft'] = result['stance_current']
        df_test_stance.loc[i,'stance_future_gpt_ft'] = result['stance_future']
    except Exception:
        df_test_stance.loc[i, 'error_gpt_ft'] = chat_completion.choices[0].message.content

# df_test_stance['error_gpt_ft'] = df_test_stance['error_gpt_ft'].apply(lambda x: json.loads(x.replace("\'", "\"")))
# df_test_stance['stance_current_gpt_ft'] = df_test_stance['error_gpt_ft'].apply(lambda x: x['stance_current'])
# df_test_stance['stance_future_gpt_ft'] = df_test_stance['error_gpt_ft'].apply(lambda x: x['stance_future'])

In [221]:
# ft v1:
'raw:', accuracy_score(df_test_stance['stance_current'], df_test_stance['stance_current_gpt_ft']), accuracy_score(df_test_stance['stance_future'], df_test_stance['stance_future_gpt_ft']), f1_score(df_test_stance['stance_current'], df_test_stance['stance_current_gpt_ft'], average='weighted'), f1_score(df_test_stance['stance_future'], df_test_stance['stance_future_gpt_ft'], average='weighted'), 'merging unclear/irrelevant:', \
accuracy_score(df_test_stance['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_current_gpt_ft'].apply(lambda x: 'unclear' if x=='irrelevant' else x)), accuracy_score(df_test_stance['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_future_gpt_ft'].apply(lambda x: 'unclear' if x=='irrelevant' else x)), f1_score(df_test_stance['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_current_gpt_ft'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted'), f1_score(df_test_stance['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_future_gpt_ft'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted')

('raw:',
 0.8017241379310345,
 0.6982758620689655,
 0.7878694581280787,
 0.6866524352766146,
 'merging unclear/irrelevant:',
 0.8448275862068966,
 0.7413793103448276,
 0.8448891625615763,
 0.7400481164682767)

In [239]:
df_test_stance['stance_current'].value_counts()

unclear          52
accommodative    40
restrictive      15
irrelevant        7
neutral           2
Name: stance_current, dtype: int64

In [241]:
confusion_matrix(df_test_stance['stance_current'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_current_gpt_ft'].apply(lambda x: 'unclear' if x=='irrelevant' else x), labels=['accommodative', 'neutral', 'restrictive', 'unclear'])

array([[33,  0,  0,  7],
       [ 0,  2,  0,  0],
       [ 0,  0, 12,  3],
       [ 7,  0,  1, 51]], dtype=int64)

In [236]:
confusion_matrix(df_test_stance['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_test_stance['stance_future_gpt1'].apply(lambda x: 'unclear' if x=='irrelevant' else x), labels=['tightening', 'tightening bias', 'no change', 'loosening bias', 'loosening', 'unclear'])

array([[11,  0,  2,  0,  0,  0],
       [ 6,  6,  4,  0,  0,  0],
       [ 0,  1, 40,  2,  1,  3],
       [ 0,  0,  6, 10,  0,  0],
       [ 0,  0,  0,  3,  4,  0],
       [ 0,  0,  3,  0,  0, 14]], dtype=int64)

In [243]:
# save results
df_test_stance.drop('error_gpt_ft', axis=1).to_excel(directory+'output/finetuning/df_test_stance_mon.xlsx', index=False)

In [179]:
# x = df_train_stance.loc[461]
messages = [
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year written by %s, complete the following two tasks. First, classify the country\'s recent or current monetary policy stance as described in the text into restrictive/neutral/accommodative/unclear/irrelevant; if it discusses monetary policy but the specific stance is not clear, assign unclear; if it does not discuss monetary policy, assign irrelevant. Second, classify the %s\'s recommended or planned near-future (next year) direction of change in monetary policy stance as described in the text into tightening/tightening bias/no change/loosening bias/loosening/unclear/irrelevant; if it discusses monetary policy stance but the direction of change is not clear, assign no change; if it does not discuss monetary policy stance, assign unclear (if it discusses monetary policy) or irrelevant (if it does not discuss monetary policy).\n%s\n\nReturn a JSON dict without additional texts: \"stance_current\", \"stance_future\".'''%(type_dict[row['type']],  type_dict[row['type']], example_dict[row['type']]),},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\nText:\n%s''' % (row['country'], row['year'], row['text'])}
        ]

In [180]:
print(messages[0]['content'] + '\n\n' + messages[1]['content'])

You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year written by country's authority, complete the following two tasks. First, classify the country's recent or current monetary policy stance as described in the text into restrictive/neutral/accommodative/unclear/irrelevant; if it discusses monetary policy but the specific stance is not clear, assign unclear; if it does not discuss monetary policy, assign irrelevant. Second, classify the country's authority's recommended or planned near-future (next year) direction of change in monetary policy stance as described in the text into tightening/tightening bias/no change/loosening bias/loosening/unclear/irrelevant; if it discusses monetary policy stance but the direction of change is not clear, assign no change; if it does not discuss monetary policy stance, assign unclear (if it discusses monetary policy) or irrelevant (if it does not discuss monetary policy).
Example 1: Country